In [1]:
print(7)

7


In [ ]:
# Step 1: Import everything we need

In [29]:
import os 
from dotenv import load_dotenv

# langchain
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter 
from langchain_community.embeddings import JinaEmbeddings

In [30]:
load_dotenv(override=True)

True

In [31]:
groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")


print("ENV VAR LOADED")

ENV VAR LOADED


In [ ]:
# Loading our data

In [32]:
DATA_FILE_PATH = os.path.join("data" , "hr_policy.txt")

In [ ]:
# Data ingestion

In [33]:
DATA_FILE_PATH = os.path.join("data", "hr_policy.txt")

loader = TextLoader(DATA_FILE_PATH, encoding="utf-8")
documents = loader.load()

print(f"Loaded file: {DATA_FILE_PATH}")
print(f"Number of documents loaded: {len(documents)}")
print(f"Total characters in document: {len(documents[0].page_content)}")
print("\n--- Preview of first 300 characters ---")
print(documents[0].page_content[:300])

Loaded file: data\hr_policy.txt
Number of documents loaded: 1
Total characters in document: 2598

--- Preview of first 300 characters ---
COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carr


LANGCHAIN DOCUMENT
Langchain processes everything in form of documents

DOCUMENTS :

PAGE CONTENT -- the actual data


METADATA - extra information about the data

In [34]:
print(documents[0].metadata)

{'source': 'data\\hr_policy.txt'}


SPLITTING OUR DATA

In [35]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

chunks = text_splitter.split_documents(documents)

print(chunks)

[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM

In [36]:
len(chunks)

9

In [37]:
chunks = text_splitter.split_documents(documents)

print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {len(chunk.page_content)} characters")

Number of chunks: 9
Chunk 1: 72 characters
Chunk 2: 451 characters
Chunk 3: 267 characters
Chunk 4: 326 characters
Chunk 5: 294 characters
Chunk 6: 339 characters
Chunk 7: 311 characters
Chunk 8: 218 characters
Chunk 9: 303 characters


NOW EACH SPILTED CHUNK IS A DOCUMENT - page content and metadata

In [14]:
print(chunks[0])

page_content='COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)' metadata={'source': 'data\\hr_policy.txt'}


In [15]:
print(chunks[5])

page_content='5. REIMBURSEMENT POLICY
Employees can claim reimbursement for approved business expenses such as travel,
client meals, and internet bills used for official work.
All reimbursement claims must be submitted with valid bills within 30 days of the expense.
Claims are processed within 10 working days after approval from the reporting manager.' metadata={'source': 'data\\hr_policy.txt'}


In [16]:
print(chunks[7].page_content)

7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.


EMBED OUR DATA


In [25]:
from langchain_community.embeddings import JinaEmbeddings

embeddings_model = JinaEmbeddings(model_name="jina-embeddings-v2-base-en")

print("EMB MODEL READY THE NAME IS ", embeddings_model.model_name)


EMB MODEL READY THE NAME IS  jina-embeddings-v2-base-en


STORE DATA IN VECTOR DB


In [20]:
import os

print(os.environ.get("PINECONE_API_KEY") is not None)



True


In [22]:
from pinecone import Pinecone

pc = Pinecone(
    api_key=os.environ["PINECONE_API_KEY"]
)

print(pc.list_indexes().names())


['rag-index']


In [26]:
print(len(embeddings_model.embed_query("Hello world")))


768


In [38]:
from langchain_pinecone import PineconeVectorStore

index = pc.Index("rag-index")

vector_store = PineconeVectorStore(
    index=index,
    embedding=embeddings_model
)

vector_store.add_documents(chunks)

print("CHUNKS ARE STORED")

CHUNKS ARE STORED


    WE NEVER STORED IT
    

In [25]:
test_query = "How many sick leaves employees get"

## SIMILARITY SEARCH 

top_matches = vector_store.similarity_search(test_query , k=2)
print(f"Query: {test_query}\n")
for i,match in enumerate(top_matches,start=1):
    print(f"--- Match {i} ---")
    print(match.page_content)
    print()

Query: How many sick leaves employees get

--- Match 1 ---
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

--- Match 2 ---
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.



#### TOOL

In [39]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})  # returns top 3 relevant chunks

def Search_hr_policy(question:str)->str:
    """
    Search the HR policy document for information about leave, work from home,
    probation, notice period, reimbursement, code of conduct, holidays, or exit process.
    
    """
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

### DATA RETRIVAL
LLM

In [12]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0.5  # creativity 
)

llm.model_name

'openai/gpt-oss-120b'

In [13]:
tr = llm.invoke("Hey what is the leave policy")

In [14]:
tr.content

'Sure!\u202fLeave policies can vary a lot from one organization to another, but most companies cover the same basic categories. Below is a quick overview of the typical elements you’ll find in a standard leave policy, along with some common details you might want to check for in your own workplace.\n\n---\n\n## 1. Types of Leave\n\n| Leave Type | Typical Purpose | Common Eligibility / Accrual | Typical Duration |\n|------------|----------------|------------------------------|------------------|\n| **Annual / Vacation Leave** | Rest, personal time, travel | Earned based on length of service (e.g., 1–2\u202fdays per month) | Usually 10‑30\u202fdays per year, often pro‑rated for new hires |\n| **Sick / Medical Leave** | Illness, injury, medical appointments | Often accrued (e.g., 1\u202fday per month) or provided as a set pool | 5‑12\u202fdays per year; some policies allow carry‑over or unlimited “self‑certified” days |\n| **Personal/Unpaid Leave** | Non‑medical personal matters | Usually

AI AGENT

3 --

LLM - BRAIN

TOOL - SUPER POWER

MEMORY - no memory

In [2]:
# Dependencies are already installed in the selected notebook kernel.

In [5]:
from langgraph.prebuilt import create_react_agent as create_agent


In [45]:
hr_assistant = create_agent(
    model=llm,
    tools=[Search_hr_policy],
    prompt=(
        "You are a friendly HR assistant working for Acme Corp. "
        "Always use the Search_hr_policy tool to look up facts before answering. "
        "If the answer isn't in the search results, say you don't know "
        "instead of guessing."
    ),
)

print("HR assistant agent is ready to answer questions!")



HR assistant agent is ready to answer questions!


C:\Users\MANOJ REDDY\AppData\Local\Temp\ipykernel_9012\3568053603.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  hr_assistant = create_agent(


In [46]:
def ask_hr_assistant(question: str) -> str:
    """Send a question to the RAG agent and print a nicely formatted answer."""
    print("=" * 60)
    print("QUESTION:", question)
    print("-" * 60)

    response = hr_assistant.invoke({"messages": [{"role": "user", "content": question}]})
    answer = response["messages"][-1].content

    print("ANSWER:", answer)
    print("=" * 60)
    print()
    return answer

In [ ]:
response = hr_assistant.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content": "tell me about the live policy"
            }
        ]
    }
)

In [48]:
response 

{'messages': [HumanMessage(content='tell me which org you work for', additional_kwargs={}, response_metadata={}, id='aacc2bc8-6501-4cfd-bf55-2dda386a5b16'),
  AIMessage(content='I’m the friendly HR assistant here at **Acme\u202fCorp**. How can I help you today?', additional_kwargs={'reasoning_content': 'User asks: "tell me which org you work for". As HR assistant for Acme Corp. Should answer: I work for Acme Corp. Should we verify via policy? Not needed. But we can answer directly.'}, response_metadata={'token_usage': {'completion_tokens': 77, 'prompt_tokens': 198, 'total_tokens': 275, 'completion_time': 0.159582369, 'completion_tokens_details': {'reasoning_tokens': 46}, 'prompt_time': 0.009122515, 'prompt_tokens_details': None, 'queue_time': 0.214461804, 'total_time': 0.168704884}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_6dedd2be22', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--01a07531-694c-7801-bbe5-856f75a9f8c0-0', usage_m

SYSTEM MESSAGE - HR ASSISNT

HUMAN MESSAGE - TELL ME ABOUT POLICIES

AI MESSAGE - HEY THESE ARE THEPLOICES

In [49]:
response["messages"][-2].content

'tell me which org you work for'

In [50]:
print(response["messages"][-1].content)

I’m the friendly HR assistant here at **Acme Corp**. How can I help you today?
